# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
import json
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = (SparkSession.builder.appName('J3_Analytics_SQL_Gold')
         .master('local[2]').config('spark.sql.shuffle.partitions', '8').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
SILVER_MAIN_PATH = '/lakehouse/silver/flights'
GOLD_ROOT = '/lakehouse/gold'
REPORT = {'spark_version': spark.version, 'silver_path': SILVER_MAIN_PATH, 'gold_tables': {}}

# Projection des seules colonnes utiles, mise en cache : chaque requête évite de relire les 57 colonnes sur HDFS.
USEFUL_COLS = ['flight_date', 'month', 'day_of_week', 'airline', 'airline_name',
               'origin_airport', 'origin_airport_name', 'origin_city', 'origin_state',
               'origin_latitude', 'origin_longitude', 'scheduled_hour', 'departure_slot',
               'cancelled', 'departure_delay', 'arrival_delay', 'departure_delay_15',
               'positive_departure_delay']
silver = spark.read.format('delta').load(SILVER_MAIN_PATH).select(*USEFUL_COLS).cache()
silver.createOrReplaceTempView('v_silver')
REPORT['silver_rows'] = spark.sql('SELECT COUNT(*) FROM v_silver').first()[0]
assert REPORT['silver_rows'] == 1943809
print('Lignes disponibles dans v_silver :', REPORT['silver_rows'])


def save_gold(df, name):
    """Écrit une petite table Gold en un seul fichier Delta, puis la relit pour contrôle."""
    path = f'{GOLD_ROOT}/{name}'
    df.coalesce(1).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(path)
    rows = spark.read.format('delta').load(path).count()
    REPORT['gold_tables'][name] = {'path': path, 'rows': rows}
    print(f'[GOLD OK] {path} : {rows} lignes')
    return path

# Indicateurs communs. Dénominateurs : tous les vols pour l'annulation ; vols non annulés au retard
# renseigné pour le taux de retard (departure_delay_15 est NULL sinon). TRY_DIVIDE évite toute division par zéro.
INDICATORS = """
    COUNT(*)                                   AS flights,
    SUM(cancelled)                             AS cancelled_flights,
    COUNT(departure_delay_15)                  AS observed_flights,
    SUM(departure_delay_15)                    AS delayed_15_flights,
    TRY_DIVIDE(SUM(cancelled), COUNT(*))       AS cancellation_rate,
    TRY_DIVIDE(SUM(departure_delay_15), COUNT(departure_delay_15)) AS delay_rate_15,
    PERCENTILE_APPROX(IF(cancelled = 0, departure_delay, NULL), 0.25) AS p25_delay,
    PERCENTILE_APPROX(IF(cancelled = 0, departure_delay, NULL), 0.50) AS median_delay,
    PERCENTILE_APPROX(IF(cancelled = 0, departure_delay, NULL), 0.75) AS p75_delay,
    PERCENTILE_APPROX(IF(cancelled = 0, departure_delay, NULL), 0.90) AS p90_delay,
    ROUND(AVG(IF(cancelled = 0, departure_delay, NULL)), 2)          AS mean_delay,
    SUM(positive_departure_delay)              AS total_delay_minutes
"""

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db897ef4-2841-42d6-9364-1720a7a3e384;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central


	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central


	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central


	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 705ms :: artifacts dl 29ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from 

26/11/19 10:36:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Lignes disponibles dans v_silver : 1943809


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
df_dist_summary = spark.sql("""
WITH observations AS (
    SELECT 'departure_delay' AS measure, departure_delay AS minutes
    FROM v_silver WHERE cancelled = 0 AND departure_delay IS NOT NULL
    UNION ALL
    SELECT 'arrival_delay', arrival_delay
    FROM v_silver WHERE cancelled = 0 AND arrival_delay IS NOT NULL
)
SELECT measure,
       COUNT(*)                              AS n,
       ROUND(AVG(minutes), 2)                AS mean,
       ROUND(STDDEV(minutes), 2)             AS stddev,
       ROUND(SKEWNESS(minutes), 2)           AS skewness,
       MIN(minutes)                          AS min,
       PERCENTILE_APPROX(minutes, 0.25)      AS p25,
       PERCENTILE_APPROX(minutes, 0.50)      AS p50,
       PERCENTILE_APPROX(minutes, 0.75)      AS p75,
       PERCENTILE_APPROX(minutes, 0.90)      AS p90,
       PERCENTILE_APPROX(minutes, 0.95)      AS p95,
       MAX(minutes)                          AS max,
       ROUND(AVG(IF(minutes >= 15, 1.0, 0.0)), 4) AS share_15_plus,
       ROUND(AVG(IF(minutes < 0, 1.0, 0.0)), 4)   AS share_early
FROM observations
GROUP BY measure
ORDER BY measure DESC
""")
df_dist_summary.show(truncate=False)
REPORT['distribution'] = {r['measure']: {k: v for k, v in r.asDict().items() if k != 'measure'}
                          for r in df_dist_summary.collect()}
dep = REPORT['distribution']['departure_delay']
REPORT['distribution_checks'] = {'mean_minus_median': round(dep['mean'] - dep['p50'], 2),
                                 'iqr': dep['p75'] - dep['p25']}
print('Écart moyenne − médiane (départ) :', REPORT['distribution_checks']['mean_minus_median'], 'min ;',
      'IQR :', REPORT['distribution_checks']['iqr'], 'min')

+---------------+-------+----+------+--------+-----+-----+----+---+----+----+------+-------------+-----------+
|measure        |n      |mean|stddev|skewness|min  |p25  |p50 |p75|p90 |p95 |max   |share_15_plus|share_early|
+---------------+-------+----+------+--------+-----+-----+----+---+----+----+------+-------------+-----------+
|departure_delay|1917902|8.98|35.84 |7.36    |-68.0|-5.0 |-2.0|7.0|34.0|65.0|1988.0|0.1825       |0.5756     |
|arrival_delay  |1913015|4.18|38.17 |6.25    |-87.0|-13.0|-5.0|8.0|34.0|65.0|1971.0|0.1863       |0.6098     |
+---------------+-------+----+------+--------+-----+-----+----+---+----+----+------+-------------+-----------+



Écart moyenne − médiane (départ) : 10.98 min ; IQR : 12.0 min


**Justification statistique :**

Sur **1 917 902 vols** non annulés au retard renseigné, le retard au départ a une **moyenne de 8,98 min** mais une **médiane de −2 min** : plus de la moitié des vols partent en avance (57,6 %). Le coefficient d'asymétrie (**skewness = 7,36**) révèle une longue traîne à droite : P90 = 34 min, P95 = 65 min, maximum 1 988 min (plus de 33 h). Quelques retards extrêmes tirent la moyenne vers le haut, à environ 11 min au-dessus de la médiane ; l'écart-type (35,8 min) est lui aussi dominé par cette traîne. Le retard à l'arrivée présente le même profil (skewness = 6,25).

Indicateurs retenus pour la couche Gold et le dashboard :
- **médiane et IQR** (P75 − P25 = 12 min) pour le retard typique et sa dispersion, peu sensibles aux valeurs extrêmes ;
- **P90** pour décrire la queue de distribution, c'est-à-dire les retards qui pèsent réellement sur les voyageurs ;
- **taux de retard ≥ 15 min** (18,25 % des vols observés), seuil de ponctualité du ministère américain des transports, avec un dénominateur explicite : vols non annulés dont le retard est renseigné ;
- **total des minutes positives de retard** pour mesurer la concentration (Pareto) : il combine volume de trafic et intensité.

La médiane est proche de zéro dans presque tous les groupes et discrimine donc peu : les comparaisons s'appuient surtout sur le taux de retard et le P90, toujours accompagnés des effectifs. La moyenne n'est conservée qu'à titre indicatif.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
GOLD_PARETO_PATH = f'{GOLD_ROOT}/airport_delay_pareto'
MIN_FLIGHTS_FOR_RATE = 5000  # classement par taux limité aux aéroports au volume suffisant

df_gold_1 = spark.sql(f"""
WITH by_airport AS (
    SELECT origin_airport AS airport, origin_airport_name AS airport_name,
           origin_city AS city, origin_state AS state,
           origin_latitude AS latitude, origin_longitude AS longitude,
           {INDICATORS}
    FROM v_silver
    GROUP BY origin_airport, origin_airport_name, origin_city, origin_state,
             origin_latitude, origin_longitude
)
SELECT *,
       p75_delay - p25_delay AS iqr_delay,
       DENSE_RANK() OVER (ORDER BY total_delay_minutes DESC) AS rank_total_minutes,
       SUM(total_delay_minutes) OVER (ORDER BY total_delay_minutes DESC, airport
                                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
         / SUM(total_delay_minutes) OVER () AS cumulative_share_minutes,
       ROW_NUMBER() OVER (ORDER BY total_delay_minutes DESC, airport)
         / COUNT(*) OVER () AS cumulative_share_airports,
       IF(flights >= {MIN_FLIGHTS_FOR_RATE},
          DENSE_RANK() OVER (PARTITION BY flights >= {MIN_FLIGHTS_FOR_RATE} ORDER BY delay_rate_15 DESC),
          NULL) AS rank_delay_rate
FROM by_airport
""").withColumn('share_before_airport', F.col('cumulative_share_minutes') -
             F.col('total_delay_minutes') / F.sum('total_delay_minutes').over(Window.partitionBy()))
# Inclure l'aéroport qui permet de franchir 80 %, puis celui qui franchit 95 %.
df_gold_1 = df_gold_1.withColumn('pareto_class', F.when(F.col('share_before_airport') < .80, 'A')
                .when(F.col('share_before_airport') < .95, 'B').otherwise('C'))

save_gold(df_gold_1, 'airport_delay_pareto')
gold_1 = spark.read.format('delta').load(GOLD_PARETO_PATH)
assert gold_1.count() == gold_1.select('airport').distinct().count(), 'Une ligne par aéroport attendue.'
assert gold_1.agg(F.sum('flights')).first()[0] == REPORT['silver_rows']
gold_1.orderBy('rank_total_minutes').select(
    'rank_total_minutes', 'airport', 'city', 'flights', 'total_delay_minutes', 'delay_rate_15',
    'median_delay', 'iqr_delay', 'p90_delay', 'cumulative_share_minutes', 'pareto_class').show(15, truncate=False)

class_a = gold_1.filter("pareto_class = 'A'")
REPORT['pareto'] = {
    'airports': gold_1.count(),
    'class_a_airports': class_a.count(),
    'class_a_share_of_airports': round(class_a.count() / gold_1.count(), 4),
    'class_a_share_minutes': float(class_a.agg(F.sum('total_delay_minutes')).first()[0] / gold_1.agg(F.sum('total_delay_minutes')).first()[0]),
    'class_a_share_of_flights': round(class_a.agg(F.sum('flights')).first()[0] / REPORT['silver_rows'], 4),
    'top10_share_minutes': round(gold_1.filter('rank_total_minutes <= 10')
                                 .agg(F.max('cumulative_share_minutes')).first()[0], 4),
    'top5_by_minutes': [r.asDict() for r in gold_1.orderBy('rank_total_minutes').limit(5)
                        .select('airport', 'city', 'flights', 'total_delay_minutes', 'delay_rate_15').collect()],
    'top5_by_rate_min_flights': [r.asDict() for r in gold_1.filter('rank_delay_rate IS NOT NULL')
                                 .orderBy('rank_delay_rate').limit(5)
                                 .select('airport', 'city', 'flights', 'delay_rate_15', 'rank_total_minutes').collect()],
    'min_flights_for_rate': MIN_FLIGHTS_FOR_RATE,
}
print(json.dumps({k: v for k, v in REPORT['pareto'].items() if not k.startswith('top5')}, indent=1))
print('Top 5 par taux de retard (≥ 5 000 vols) :')
gold_1.filter('rank_delay_rate IS NOT NULL').orderBy('rank_delay_rate').select(
    'rank_delay_rate', 'airport', 'city', 'flights', 'delay_rate_15', 'median_delay', 'rank_total_minutes').show(5)

[GOLD OK] /lakehouse/gold/airport_delay_pareto : 322 lignes


+------------------+-------+-----------------+-------+-------------------+-------------------+------------+---------+---------+------------------------+------------+
|rank_total_minutes|airport|city             |flights|total_delay_minutes|delay_rate_15      |median_delay|iqr_delay|p90_delay|cumulative_share_minutes|pareto_class|
+------------------+-------+-----------------+-------+-------------------+-------------------+------------+---------+---------+------------------------+------------+
|1                 |ORD    |Chicago          |103590 |1637645.0          |0.23712415444061466|-1.0        |17.0     |51.0     |0.0725194488797324      |A           |
|2                 |ATL    |Atlanta          |125524 |1227733.0          |0.16797856171506279|-1.0        |9.0      |28.0     |0.12688686094489945     |A           |
|3                 |DFW    |Dallas-Fort Worth|87449  |1174415.0          |0.22097551928783382|-1.0        |15.0     |42.0     |0.17889320453956795     |A           |
|4  

{
 "airports": 322,
 "class_a_airports": 45,
 "class_a_share_of_airports": 0.1398,
 "class_a_share_minutes": 0.8027538477405317,
 "class_a_share_of_flights": 0.7834,
 "top10_share_minutes": 0.3991,
 "min_flights_for_rate": 5000
}
Top 5 par taux de retard (≥ 5 000 vols) :


+---------------+-------+---------+-------+-------------------+------------+------------------+
|rank_delay_rate|airport|     city|flights|      delay_rate_15|median_delay|rank_total_minutes|
+---------------+-------+---------+-------+-------------------+------------+------------------+
|              1|    MDW|  Chicago|  29467|0.24251321197886083|         1.0|                16|
|              2|    DEN|   Denver|  70792| 0.2395448120279281|         0.0|                 4|
|              3|    ORD|  Chicago| 103590|0.23712415444061466|        -1.0|                 1|
|              4|    BWI|Baltimore|  31437|0.23622881355932204|         0.0|                14|
|              5|    DAL|   Dallas|  21611| 0.2323081237730205|         1.0|                26|
+---------------+-------+---------+-------+-------------------+------------+------------------+
only showing top 5 rows


### Interprétation — Pareto des aéroports

- **45 aéroports sur 322 (14,0 %)** concentrent 80,28 % des minutes de retard au départ ; ils assurent aussi **78,34 % des vols**. Les 10 premiers cumulent 39,9 % des minutes.
- Cette concentration reflète d'abord celle du trafic : le total de minutes additionne volume et intensité. Il ne désigne pas, à lui seul, les aéroports les moins ponctuels.
- La comparaison des deux classements le montre. **Atlanta (ATL)** est 2e en minutes cumulées, mais son taux de retard (16,8 %) est inférieur au taux global (18,25 %). À l'inverse, **Chicago Midway (MDW)** a le plus fort taux parmi les aéroports d'au moins 5 000 vols (24,3 %), mais n'est que 16e en minutes. **Chicago O'Hare (ORD)** cumule les deux : 1er en minutes et 3e en taux.
- Le seuil de 5 000 vols évite de placer en tête de petits aéroports dont le taux repose sur peu d'observations.

Table `/lakehouse/gold/airport_delay_pareto` : une ligne par aéroport d'origine, avec ses coordonnées pour la carte du dashboard. Trois aéroports n'ont pas de coordonnées dans le référentiel (ECP, PBG et UST : 1 662 vols au total) : ils restent dans les classements mais seront filtrés sur la carte.

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = f'{GOLD_ROOT}/daily_delay_trend'

# Les quatre mois ne sont pas contigus : LAG et moyenne glissante sont calculés à l'intérieur de chaque mois
# (PARTITION BY month) pour ne pas comparer le 31 janvier au 1er avril.
df_gold_2 = spark.sql(f"""
WITH by_day AS (
    SELECT flight_date, month, day_of_week, {INDICATORS}
    FROM v_silver
    GROUP BY flight_date, month, day_of_week
)
SELECT *,
       LAG(delay_rate_15) OVER w                          AS prev_day_delay_rate,
       100.0 * (delay_rate_15 - LAG(delay_rate_15) OVER w) AS delay_rate_change_pts,
       100.0 * TRY_DIVIDE(total_delay_minutes - LAG(total_delay_minutes) OVER w,
                  LAG(total_delay_minutes) OVER w)        AS total_minutes_change_pct,
       AVG(delay_rate_15) OVER (PARTITION BY month ORDER BY flight_date
                                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS delay_rate_7d_avg
FROM by_day
WINDOW w AS (PARTITION BY month ORDER BY flight_date)
""")

save_gold(df_gold_2, 'daily_delay_trend')
gold_2 = spark.read.format('delta').load(GOLD_TIME_PATH)
assert gold_2.count() == 31 + 30 + 31 + 30, 'Un enregistrement par jour des quatre mois attendu.'
assert gold_2.filter('prev_day_delay_rate IS NULL').count() == 4, 'Seul le premier jour de chaque mois est sans LAG.'
gold_2.orderBy('flight_date').select('flight_date', 'flights', 'delay_rate_15', 'prev_day_delay_rate',
                                     'delay_rate_change_pts', 'delay_rate_7d_avg', 'cancellation_rate',
                                     'median_delay').show(20, truncate=False)

df_month = spark.sql(f"SELECT month, {INDICATORS} FROM v_silver GROUP BY month ORDER BY month")
df_month.select('month', 'flights', 'delay_rate_15', 'cancellation_rate', 'median_delay', 'p90_delay',
                'mean_delay').show()
REPORT['monthly'] = [r.asDict() for r in df_month.select(
    'month', 'flights', 'delay_rate_15', 'cancellation_rate', 'median_delay', 'p90_delay', 'mean_delay').collect()]
REPORT['worst_days'] = [r.asDict() for r in gold_2.orderBy(F.desc('delay_rate_15')).limit(5).select(
    F.col('flight_date').cast('string').alias('flight_date'), 'flights', 'delay_rate_15',
    'cancellation_rate', 'median_delay').collect()]
REPORT['largest_day_increase'] = gold_2.orderBy(F.desc('delay_rate_change_pts')).select(
    F.col('flight_date').cast('string').alias('flight_date'), 'prev_day_delay_rate', 'delay_rate_15',
    'delay_rate_change_pts').first().asDict()
print('Jours au plus fort taux de retard :')
gold_2.orderBy(F.desc('delay_rate_15')).select('flight_date', 'flights', 'delay_rate_15',
                                               'cancellation_rate', 'median_delay').show(5)

[GOLD OK] /lakehouse/gold/daily_delay_trend : 122 lignes


+-----------+-------+-------------------+-------------------+---------------------+-------------------+---------------------+------------+
|flight_date|flights|delay_rate_15      |prev_day_delay_rate|delay_rate_change_pts|delay_rate_7d_avg  |cancellation_rate    |median_delay|
+-----------+-------+-------------------+-------------------+---------------------+-------------------+---------------------+------------+
|2015-01-01 |13950  |0.21514387422129932|NULL               |NULL                 |0.21514387422129932|0.03340501792114695  |-1.0        |
|2015-01-02 |16741  |0.26977675321523903|0.21514387422129932|5.46328789939397     |0.24246031371826918|0.015351532166537243 |0.0         |
|2015-01-03 |15434  |0.4093888631397736 |0.26977675321523903|13.961210992453454   |0.2981031635254373 |0.02144615783335493  |8.0         |
|2015-01-04 |16352  |0.46717758653181735|0.4093888631397736 |5.778872339204377    |0.34037176927703233|0.026479941291585127 |12.0        |
|2015-01-05 |16548  |0.3466

+-----+-------+-------------------+--------------------+------------+---------+----------+
|month|flights|      delay_rate_15|   cancellation_rate|median_delay|p90_delay|mean_delay|
+-----+-------+-------------------+--------------------+------------+---------+----------+
|    1| 469968| 0.1983139222596324|0.025495352875089368|        -2.0|     37.0|      9.73|
|    4| 485151|0.16589441796305274|0.009316686969623891|        -2.0|     29.0|      7.68|
|    7| 520718| 0.2126913116965684|0.009229563794606677|        -1.0|     42.0|     11.37|
|   11| 467972|0.15031303075492097|0.009827511047669518|        -2.0|     26.0|      6.92|
+-----+-------+-------------------+--------------------+------------+---------+----------+



Jours au plus fort taux de retard :


+-----------+-------+-------------------+--------------------+------------+
|flight_date|flights|      delay_rate_15|   cancellation_rate|median_delay|
+-----------+-------+-------------------+--------------------+------------+
| 2015-01-04|  16352|0.46717758653181735|0.026479941291585127|        12.0|
| 2015-01-03|  15434| 0.4093888631397736| 0.02144615783335493|         8.0|
| 2015-01-06|  15315| 0.3510723860589812|0.025791707476330396|         3.0|
| 2015-01-05|  16548|0.34667659653695776|0.026287164612037707|         4.0|
| 2015-07-18|  14892|0.33921850407171694|0.018734891216760675|         1.0|
+-----------+-------+-------------------+--------------------+------------+
only showing top 5 rows


### Interprétation — dynamique temporelle

- Le taux de retard ≥ 15 min varie selon le mois : **21,3 % en juillet**, 19,8 % en janvier, 16,6 % en avril et **15,0 % en novembre**. Janvier se distingue surtout par son taux d'annulation (2,5 %, contre environ 0,9 % les autres mois).
- Les journées les plus difficiles sont **du 3 au 6 janvier** (de 34,7 % à 46,7 % de vols retardés ; médiane de +12 min le 4 janvier). La plus forte hausse d'un jour sur l'autre est observée le **18 juillet** : +14,8 points (de 19,1 % à 33,9 %). Les données agrégées ici ne permettent pas d'en attribuer la cause.
- `LAG` et la moyenne glissante des taux journaliers sur 7 jours (poids égal par jour) sont calculés **à l'intérieur de chaque mois** : l'extrait n'est pas continu et comparer le 31 janvier au 1er avril n'aurait pas de sens. Le premier jour de chaque mois n'a donc pas de valeur précédente.
- Chaque saison n'est représentée que par un mois : ces écarts restent descriptifs. Leur significativité sera testée au Jour 4 (Kruskal-Wallis).

Table `/lakehouse/gold/daily_delay_trend` : une ligne par jour (122 jours).

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
# Gold #3 : compagnies par mois, avec rang dans le mois (ROW_NUMBER) ; la compagnie de rang 1 a le plus fort taux.
df_gold_3 = spark.sql(f"""
WITH by_airline_month AS (
    SELECT month, airline, airline_name, {INDICATORS}
    FROM v_silver
    GROUP BY month, airline, airline_name
)
SELECT *,
       p75_delay - p25_delay AS iqr_delay,
       ROW_NUMBER() OVER (PARTITION BY month ORDER BY delay_rate_15 DESC, airline) AS rank_in_month
FROM by_airline_month
""")
save_gold(df_gold_3, 'airline_monthly_performance')
gold_3 = spark.read.format('delta').load(f'{GOLD_ROOT}/airline_monthly_performance')
assert gold_3.agg(F.sum('flights')).first()[0] == REPORT['silver_rows']
print('Compagnie au plus fort taux de retard, pour chaque mois :')
gold_3.filter('rank_in_month = 1').orderBy('month').select(
    'month', 'airline', 'airline_name', 'flights', 'delay_rate_15', 'median_delay', 'p90_delay',
    'cancellation_rate').show(truncate=False)
REPORT['worst_airline_by_month'] = [r.asDict() for r in gold_3.filter('rank_in_month = 1').orderBy('month')
                                    .select('month', 'airline', 'airline_name', 'flights', 'delay_rate_15').collect()]

# Gold #4 : profil horaire selon l'heure de départ prévue, pour l'axe « créneaux » de la problématique.
df_gold_4 = spark.sql(f"""
SELECT month, scheduled_hour, departure_slot, {INDICATORS}
FROM v_silver
GROUP BY month, scheduled_hour, departure_slot
""")
save_gold(df_gold_4, 'hourly_delay_profile')
df_slot = spark.sql(f"SELECT departure_slot, {INDICATORS} FROM v_silver GROUP BY departure_slot")
df_slot.orderBy(F.desc('delay_rate_15')).select('departure_slot', 'flights', 'delay_rate_15', 'median_delay',
                                                'p90_delay', 'cancellation_rate').show()
REPORT['slots'] = [r.asDict() for r in df_slot.select('departure_slot', 'flights', 'delay_rate_15',
                                                       'median_delay', 'p90_delay', 'cancellation_rate').collect()]

# Compaction de Silver : beaucoup de petits fichiers coûtent en métadonnées NameNode et en tâches Spark.
files_before = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").first()['numFiles']
optimize_result = spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`")
optimize_result.show(truncate=False)
detail_after = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").first()
metrics = optimize_result.first()['metrics']
REPORT['optimize'] = {'files_before': files_before, 'files_after': detail_after['numFiles'],
                      'size_bytes_after': detail_after['sizeInBytes'],
                      'files_added': metrics['numFilesAdded'], 'files_removed': metrics['numFilesRemoved']}
assert spark.read.format('delta').load(SILVER_MAIN_PATH).count() == REPORT['silver_rows']
print('Fichiers actifs dans Silver :', files_before, '->', detail_after['numFiles'])

silver.unpersist()
Path('/workspace/reports/j3_gold.json').write_text(
    json.dumps(REPORT, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Jour 3 validé ; rapport : reports/j3_gold.json')

[GOLD OK] /lakehouse/gold/airline_monthly_performance : 54 lignes


Compagnie au plus fort taux de retard, pour chaque mois :


+-----+-------+----------------------------+-------+-------------------+------------+---------+--------------------+
|month|airline|airline_name                |flights|delay_rate_15      |median_delay|p90_delay|cancellation_rate   |
+-----+-------+----------------------------+-------+-------------------+------------+---------+--------------------+
|1    |MQ     |American Eagle Airlines Inc.|29900  |0.29063450673620167|-1.0        |60.0     |0.07652173913043478 |
|4    |F9     |Frontier Airlines Inc.      |7148   |0.22630098452883263|-2.0        |41.0     |0.005316172355903749|
|7    |UA     |United Air Lines Inc.       |46478  |0.297174527276679  |3.0         |66.0     |0.010069280089504713|
|11   |F9     |Frontier Airlines Inc.      |7763   |0.21288986670117768|-2.0        |43.0     |0.004637382455236378|
+-----+-------+----------------------------+-------+-------------------+------------+---------+--------------------+



[GOLD OK] /lakehouse/gold/hourly_delay_profile : 96 lignes


+--------------+-------+-------------------+------------+---------+--------------------+
|departure_slot|flights|      delay_rate_15|median_delay|p90_delay|   cancellation_rate|
+--------------+-------+-------------------+------------+---------+--------------------+
|          Soir| 426528|  0.263832047757667|         0.0|     51.0|0.015806699677395154|
|    Après-midi| 714789|0.21215002132353253|        -1.0|     39.0|0.012585532233987931|
|         Matin| 754527|0.11549896977791499|        -3.0|     18.0|0.012646333398274682|
|          Nuit|  47965|0.07199290210824284|        -3.0|      8.0|0.013072031689773793|
+--------------+-------+-------------------+------------+---------+--------------------+



+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                         |metrics                                                                                                                                                                                 |
+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/flights|{1, 8, {75604074, 75604074, 7.5604074E7, 1, 75604074}, {9479535, 9512351, 9497361.125, 8, 75978889}, 1, NULL, NULL, 1, 1, 8, 0, false, 0, 0, 1795084709459, 0, 2, 0, NULL, NULL, 57, 32}|
+---------------------------------------------+---------------------

Fichiers actifs dans Silver : 8 -> 1
Jour 3 validé ; rapport : reports/j3_gold.json


### Interprétation — compagnies, créneaux et compaction

**Compagnies (`/lakehouse/gold/airline_monthly_performance`, 54 lignes).** On attendrait 56 lignes (14 compagnies × 4 mois), mais US Airways (US) n'a aucun vol en juillet ni en novembre dans la source ; cette absence est constatée dans l’extrait. `ROW_NUMBER() OVER (PARTITION BY month ...)` classe les compagnies dans chaque mois. La compagnie au plus fort taux de retard change selon le mois : American Eagle (MQ) en janvier (29,1 %, avec 7,7 % d'annulations), Frontier (F9) en avril et novembre (22,6 % et 21,3 %), United (UA) en juillet (29,7 %). Les effectifs diffèrent fortement : 7 148 vols pour Frontier en avril, 46 478 pour United en juillet. On ne peut donc pas conclure à une compagnie structurellement moins ponctuelle sur ces seuls classements.

**Créneaux (`/lakehouse/gold/hourly_delay_profile`, 96 lignes = 4 mois × 24 heures).** Le taux de retard passe de **11,5 % le matin** à 21,2 % l'après-midi et **26,4 % le soir** ; le P90 passe de 18 à 39 puis 51 min. Ce gradient est compatible avec une accumulation des retards au fil de la journée, association étudiée au Jour 4 (Mann-Whitney matin/soir), sans démontrer un mécanisme causal. Les créneaux ne regroupent toutefois pas les mêmes aéroports ni les mêmes compagnies.

**Compaction.** `OPTIMIZE` réécrit les 8 fichiers actifs de Silver en un seul, dans une transaction Delta. Les micro-fichiers pénalisent HDFS car le NameNode garde en mémoire un objet par fichier et par bloc (chaque fichier et bloc a un coût en métadonnées) : c'est le nombre de fichiers qui le charge, pas leur volume. Chaque ouverture de fichier coûte aussi un appel au NameNode et une connexion à un DataNode, et Spark doit gérer les ouvertures et les portions de lecture ; plusieurs petits fichiers peuvent être regroupés dans une même partition. Les anciens fichiers restent sur HDFS pour le Time Travel tant qu'un `VACUUM` ne les supprime pas ; nous ne l'exécutons pas, pour conserver l'historique. Les tables Gold sont écrites avec `coalesce(1)` : un seul fichier chacune.

Preuves : `reports/j3_gold.json`.

In [6]:
spark.stop()